# Setup and Environment Verification

**Objective:** Confirm that all dependencies are installed, the GPU is accessible,
the MOT17 dataset is present and correctly formatted, and the `benchmark` package
can perform a minimal end-to-end inference + metrics pass.

**Success criteria:** Every cell below runs to completion without an exception.
The final cell prints a summary of all checks.

In [ ]:
# Package version inventory for reproducibility records
import importlib

packages = [
    "ultralytics",
    "motmetrics",
    "cv2",
    "pandas",
    "matplotlib",
    "torch",
    "numpy",
    "psutil",
]

for name in packages:
    mod = importlib.import_module(name)
    ver = getattr(mod, "__version__", "unknown")
    print(f"{name:20s} {ver}")

In [ ]:
# Hardware availability: GPU, Hailo-8L, or CPU-only
import os
import subprocess
import torch

cuda_ok = torch.cuda.is_available()

# Detect active device profile to report Hailo backend when relevant
_profile_path = os.environ.get("DEVICE_PROFILE", "")
_backend = "cpu"
if _profile_path:
    import yaml
    with open(_profile_path) as _f:
        _backend = yaml.safe_load(_f).get("backend", "cpu")

if _backend == "hailo":
    # Hailo Python bindings (hailo_platform) are not required — the runner uses
    # ultralytics which calls into the system HailoRT daemon directly.
    # Verify readiness by querying the device via hailortcli instead.
    result = subprocess.run(
        ["hailortcli", "fw-control", "identify"],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        # Extract firmware version from output if present
        fw_line = next(
            (l for l in result.stdout.splitlines() if "firmware" in l.lower()),
            result.stdout.splitlines()[0] if result.stdout.strip() else ""
        )
        print(f"Backend        : Hailo-8L")
        print(f"Device         : {fw_line.strip()}")
    else:
        print("Backend        : Hailo-8L  ⚠ device not accessible")
        print(f"  hailortcli error: {result.stderr.strip() or 'no output'}")
        print("  Check: is the Hailo HAT connected and HailoRT service running?")
elif cuda_ok:
    dev     = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"CUDA available : True")
    print(f"Device         : {dev}")
    print(f"VRAM           : {vram_gb:.1f} GB")
else:
    print("Backend        : CPU-only")
    print(f"Profile        : {_profile_path or 'desktop fallback (no DEVICE_PROFILE set)'}")

In [ ]:
# benchmark package import and path resolution
from pathlib import Path
from benchmark.config import DATA_ROOT, RESULTS_RAW, SEQUENCES, SEQ_SUFFIX

print(f"DATA_ROOT    : {DATA_ROOT}")
print(f"RESULTS_RAW  : {RESULTS_RAW}")

missing = [p for p in [DATA_ROOT, RESULTS_RAW] if not p.exists()]
if missing:
    readme = Path("../README.md").resolve()
    raise FileNotFoundError(
        "\n\nMissing required paths:\n"
        + "".join(f"  {p}\n" for p in missing)
        + f"\nSee README.md for dataset setup instructions:\n  {readme}\n"
        + "  → Section: Prerequisites > Dataset"
    )

print("Paths OK")

In [ ]:
# MOT17 data integrity: verify img1/, gt/gt.txt, seqinfo.ini for each sequence
from benchmark.mot_gt import load_gt, load_seqinfo

checks = []
for seq in SEQUENCES:
    seq_dir = DATA_ROOT / f"{seq}-{SEQ_SUFFIX}"
    assert (seq_dir / "img1").exists(),        f"img1/ missing for {seq}"
    assert (seq_dir / "gt" / "gt.txt").exists(), f"gt.txt missing for {seq}"
    assert (seq_dir / "seqinfo.ini").exists(),  f"seqinfo.ini missing for {seq}"

    info = load_seqinfo(seq_dir)
    gt   = load_gt(seq_dir)
    n_frames = len(list((seq_dir / "img1").glob("*.jpg")))

    print(f"{seq}: {info['seqLength']} frames declared | {n_frames} jpg files | "
          f"{len(gt)} GT rows | {gt['track_id'].nunique()} tracks")
    checks.append(len(gt) > 0)

assert all(checks), "One or more sequences returned empty GT"
print("Data integrity OK")

In [ ]:
# GT format validation: unique class_id must be only {1} after filter
from pathlib import Path
import pandas as pd

seq_dir = DATA_ROOT / f"MOT17-09-{SEQ_SUFFIX}"
gt = load_gt(seq_dir)

# load_gt does not preserve class_id; reload raw to inspect
_cols = ["frame_id", "track_id", "x", "y", "w", "h", "conf", "class_id", "visibility"]
raw = pd.read_csv(seq_dir / "gt" / "gt.txt", header=None, names=_cols)

print("Raw class_id values:", sorted(raw["class_id"].unique()))
print("Raw conf values:",     sorted(raw["conf"].unique()))
print("Filtered GT sample:")
print(gt.head(5).to_string(index=False))

In [ ]:
# Model smoke test: load the smallest profile variant, track one frame, print detection count
import cv2
from ultralytics import YOLO
from benchmark.config import CONF, CLASSES, TRACKER
from benchmark.device_profile import load_profile
import os

profile = load_profile(os.environ.get("DEVICE_PROFILE"))
model_name = profile.model_variants[0]   # smallest variant in the active profile

seq_dir    = DATA_ROOT / f"MOT17-09-{SEQ_SUFFIX}"
frame_path = sorted((seq_dir / "img1").glob("*.jpg"))[0]
frame_bgr  = cv2.imread(str(frame_path))

model   = YOLO(f"models/{model_name}")
results = model.track(
    frame_bgr,
    persist=True,
    tracker=TRACKER,
    conf=CONF,
    classes=CLASSES,
    imgsz=640,
    verbose=False,
)

boxes = results[0].boxes
n = len(boxes) if boxes is not None else 0
print(f"{model_name} detections on frame 1: {n}")
print("Model smoke test OK")

In [ ]:
# motmetrics smoke test: trivial 3-frame accumulator returns a finite MOTA
# Uses benchmark._iou_distance instead of mm.distances.iou_matrix (removed in NumPy 2.0)
import math
import motmetrics as mm
import numpy as np
from benchmark.metrics import _iou_distance

acc = mm.MOTAccumulator(auto_id=False)

# Frame 1: one GT track, one prediction with high overlap
acc.update([1], [101], _iou_distance(
    np.array([[10., 10., 50., 50.]]),   # gt xywh
    np.array([[12., 12., 50., 50.]]),   # pred xywh
), frameid=1)

# Frame 2: same track continues
acc.update([1], [101], _iou_distance(
    np.array([[15., 10., 50., 50.]]),
    np.array([[16., 11., 50., 50.]]),
), frameid=2)

# Frame 3: GT present, no prediction (miss)
acc.update([1], [], np.empty((1, 0)), frameid=3)

mh      = mm.metrics.create()
summary = mh.compute(acc, metrics=["mota", "idf1", "num_switches"], name="test")
mota    = float(summary.loc["test", "mota"])

assert math.isfinite(mota), f"MOTA is not finite: {mota}"
print(f"Smoke test MOTA={mota:.3f}  IDF1={float(summary.loc['test','idf1']):.3f}  "
      f"IDSW={int(summary.loc['test','num_switches'])}")
print("motmetrics smoke test OK")

In [ ]:
# Summary
print("=" * 50)
print("All setup checks passed — environment is ready.")
print("=" * 50)

## Next steps

- Run `notebooks/01_experiment1_profiling.ipynb` to collect the 3×3 profiling matrix.
- If any cell above raised an exception, fix the dependency or path issue before proceeding.